# Frontload-cl Colab smoke (1×A100)

Checks whether **one A100** can hold the real per-rank microbatch (`24×4096`) for OLMo2-370M with FlashAttention-2 and `torch.compile`.

This is **not** the platform 8×A100 run and does **not** exercise the primer/control curriculum or `s3://edullm-data`.

**Runtime → Change runtime type → GPU** (A100 if you have Pro). Then run all cells.

Repo path used below: `/content/OLMo-core` on the branch `edullm/frontload-cl`.

## 1. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime (Runtime → Change runtime type)"
props = torch.cuda.get_device_properties(0)
print(f"device: {props.name}")
print(f"memory: {props.total_memory / 1024**3:.1f} GiB")
print(f"capability: {props.major}.{props.minor}")
print(f"torch: {torch.__version__}  cuda: {torch.version.cuda}")
if props.major < 8:
    print("WARNING: FlashAttention-2 wants SM80+ (A100). Use --attn-backend torch on older GPUs.")

## 2. Clone the branch

Uses a public HTTPS clone. If the branch is not on GitHub yet, zip this repo from your laptop, upload to Colab, unzip to `/content/OLMo-core`, and skip the clone cell.

Private clone alternative:
```
from google.colab import userdata
token = userdata.get('GH_TOKEN')  # store a read-only PAT in Colab secrets
!git clone --depth 1 --branch edullm/frontload-cl https://{token}@github.com/edu-llm/OLMo-core.git /content/OLMo-core
```

In [ ]:
import os
from pathlib import Path

REPO = Path("/content/OLMo-core")
BRANCH = "edullm/frontload-cl"
REMOTE = "https://github.com/edu-llm/OLMo-core.git"

if not (REPO / ".edullm" / "frontload_cl" / "colab_smoke.py").is_file():
    !git clone --depth 1 --branch {BRANCH} {REMOTE} {REPO}
else:
    print(f"already present: {REPO}")

%cd {REPO}
!git rev-parse --short HEAD
!ls .edullm/frontload_cl/colab_smoke.py

## 3. Install OLMo-core + try FlashAttention-2

Colab's torch build may not match the platform pin (`torch==2.9.0`). We install the package editable, then try a FA2 wheel; on failure the microbench can use `--attn-backend torch`.

In [ ]:
import subprocess
import sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("-e", ".[all]")

ATTN = "flash_2"
try:
    import flash_attn  # noqa: F401
    print("flash_attn already importable:", flash_attn.__version__)
except Exception:
    # Best-effort: platform image uses FA 2.8.3 + torch 2.9 cu12. Colab often differs.
    try:
        pip("flash-attn==2.8.3", "--no-build-isolation")
        import flash_attn
        print("installed flash_attn", flash_attn.__version__)
    except Exception as exc:
        print("flash-attn install failed; falling back to torch SDPA:", type(exc).__name__, exc)
        ATTN = "torch"

print("ATTN_BACKEND =", ATTN)

## 4. Microbench (the important cell)

Same shape as one rank on `gpu-8xa100`: **24 sequences × 4096**. Success = a few steps complete without OOM; note `peak_mem_gib`.

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ATTN set in the install cell; override here if needed: ATTN = "torch"
!python .edullm/frontload_cl/colab_smoke.py gpu-info
!python .edullm/frontload_cl/colab_smoke.py microbench --steps 3 --attn-backend {ATTN}

## 5. Optional: synthetic data → short Trainer fit

Exercises composable data loader + `TransformerTrainModule` on a **flat** mix (not primer/control). Skip if the microbench already answered your question.

In [ ]:
!python .edullm/frontload_cl/colab_smoke.py write-data --out /content/frontload-synth
!python .edullm/frontload_cl/colab_smoke.py train --data /content/frontload-synth --steps 5 --attn-backend {ATTN}

## How to read the result

| Outcome | Meaning |
| --- | --- |
| `ok: true`, peak mem well under ~40 GiB | Single-rank shape looks fine for A100 |
| CUDA OOM | Lower `--sequences` temporarily; do **not** lower the platform 8-GPU microbatch without changing the experiment |
| flash_2 import / kernel error | Re-run microbench with `--attn-backend torch`, or fix the FA2 wheel for Colab's torch |

Still needed later on the platform: 8-way HSDP/NCCL, real `frontload-cl-10b-v1`, and `.edullm/run-smoke.yaml`.